# Biomedical Hybrid IR + Patient-Review Sentiment
CAP 6776 • Juan D. Yepes

Kaggle: enable Internet; choose GPU for dense/transformer runs. Outputs are fresh only when the run cells execute. Ratings are weak sentiment labels; this notebook provides no medical advice. See docs/class-project-extension.md.

In [ ]:
import os, sys, subprocess
from pathlib import Path
assert (3, 11) <= sys.version_info[:2] < (3, 13), "Use Python 3.11/3.12 as required by pyproject.toml"
ROOT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "/content")
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / "biomedical-hybrid-ir"
BRANCH = "data/hf-kaggle-sentiment"
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                    "https://github.com/Arungharami/biomedical-hybrid-ir.git", str(REPO)], check=True)
else:
    active = subprocess.check_output(["git", "-C", str(REPO), "branch", "--show-current"], text=True).strip()
    assert active == BRANCH, f"Existing checkout is {active}; use a fresh working directory"
os.chdir(REPO)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True))
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

In [ ]:
import biomedical_ir
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_sentiment.py"], check=True)

## 1. Patient-review baseline
Downloads pinned data, audits duplicates, trains on train, tunes on dev, and evaluates held-out test. No raw review text is exported.

In [ ]:
subprocess.run([sys.executable, "scripts/run_sentiment.py", "--output", "results/sentiment-kaggle"], check=True)

## 2. Optional transformer comparison
Enable this cell for the pretrained Twitter-domain baseline. Compare only matching dataset SHA, test hashes and budgets.

In [ ]:
RUN_TRANSFORMER = False
if RUN_TRANSFORMER:
    subprocess.run([sys.executable, "scripts/run_sentiment.py", "--transformer",
                    "--output", "results/sentiment-transformer-kaggle"], check=True)

## 3. Optional full IR reproduction
This runs all existing retrieval methods and overwrites tracked historical result files in the temporary checkout. --force prevents cached historical outputs from being reported as new experiments.

In [ ]:
RUN_FULL_IR = False
if RUN_FULL_IR:
    subprocess.run([sys.executable, "scripts/reproduce.py", "--force"], check=True)

In [ ]:
import json
metrics = json.loads(Path("results/sentiment-kaggle/metrics.json").read_text())
for model, values in metrics.items():
    print(model, "Macro-F1:", values["report"]["macro avg"]["f1-score"])
    print("Confusion matrix, label order:", values["label_order"], values["confusion_matrix"])

In [ ]:
import shutil
archive = shutil.make_archive(str(ROOT / "biomedical-ir-kaggle-results"), "zip", REPO / "results")
print("Download:", archive)
print("Archive also contains historical IR outputs unless RUN_FULL_IR was enabled; use manifests to distinguish them.")